# Task 1.2: From Notebook to Production — Image Caption Generation
## Notebook 03: Inference Decoding (Beam vs. Greedy) & Spatial Attention Heatmap Visualizations

This notebook covers:
1. **Production Inference Interface**: Using `CaptionPredictor` for fast caption generation.
2. **Decoding Algorithm Comparison**: Comparing Greedy Search against Beam Search ($k=1, 3, 5, 10$).
3. **Spatial Attention Heatmap Extraction**: Visualizing where the model focuses for each generated word token.
4. **Qualitative Sample Gallery**: End-to-end demonstration on diverse test images.

In [ ]:
import sys
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch

sys.path.append("..")
from src.config import get_default_config
from src.inference.predictor import CaptionPredictor
from src.evaluation.visualization import plot_attention_heatmaps
from src.data.vocabulary import Vocabulary
from src.data.dataset import parse_flickr8k_captions

cfg = get_default_config()
vocab = Vocabulary.load(cfg.paths.vocab_path) if cfg.paths.vocab_path.exists() else None
predictor = CaptionPredictor(vocab=vocab, config=cfg)
print("CaptionPredictor initialized successfully.")

### 1. Greedy vs. Beam Search Caption Generation

In [ ]:
# Pick a test image from dataset
img_files = list(cfg.paths.images_dir.glob("*.jpg"))
test_image_path = img_files[0] if img_files else None

if test_image_path:
    raw_image = Image.open(test_image_path).convert("RGB")
    
    # Generate with Greedy Search
    greedy_res = predictor.predict(raw_image, method="greedy")
    
    # Generate with Beam Search (k=3, k=5)
    beam3_res = predictor.predict(raw_image, method="beam", beam_width=3)
    beam5_res = predictor.predict(raw_image, method="beam", beam_width=5)
    
    print(f"Image: {test_image_path.name}")
    print(f"Greedy Search:     {greedy_res.caption}")
    print(f"Beam Search (k=3): {beam3_res.caption} (Score: {beam3_res.confidence_score:.2f})")
    print(f"Beam Search (k=5): {beam5_res.caption} (Score: {beam5_res.confidence_score:.2f})")

### 2. Rendering Step-by-Step Spatial Attention Heatmaps

In [ ]:
if test_image_path and beam5_res.tokens:
    fig = plot_attention_heatmaps(
        image=raw_image,
        words=beam5_res.tokens,
        alphas=beam5_res.attention_weights,
        smooth=True
    )
    plt.show()